In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import classification_report

from xgboost import XGBClassifier

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/raw/unified_dataset.csv", low_memory=False)

# =========================
# CLEAN DATA
# =========================
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# =========================
# KEEP ONLY MALICIOUS
# =========================
df = df[df['binary_label'] == 'MALICIOUS']

print("Attack dataset shape:", df.shape)

# =========================
# CLEAN LABELS
# =========================
df['attack_type'] = df['attack_type'].astype(str).str.upper().str.strip()
df = df[~df['attack_type'].isin(['0', '1', 'BENIGN', ''])]

# =========================
# ATTACK GROUPING
# =========================
def map_attack(x):

    if 'DDOS' in x:
        return 'DDoS'
    elif 'DOS' in x:
        return 'DoS'
    elif 'SCAN' in x or 'RECON' in x:
        return 'Recon'
    elif 'PATATOR' in x or 'BRUTE' in x:
        return 'BruteForce'
    elif 'XSS' in x or 'SQL' in x or 'INJECTION' in x:
        return 'WebAttack'
    elif 'BOT' in x:
        return 'Botnet'
    elif 'MIRAI' in x or 'IOT' in x:
        return 'IoT'
    elif 'MITM' in x:
        return 'MITM'
    elif 'BACKDOOR' in x or 'MALWARE' in x:
        return 'Malware'
    else:
        return 'Other'

df['attack_type'] = df['attack_type'].apply(map_attack)

print("\n✅ Final attack classes:\n")
print(df['attack_type'].value_counts())

# =========================
# 🔥 BALANCING (IMPROVED)
# =========================
max_samples = 150000  # increased for better learning

balanced_df = []

for attack in df['attack_type'].unique():
    subset = df[df['attack_type'] == attack]

    # keep small classes fully, trim large ones
    if len(subset) > max_samples:
        subset = subset.sample(max_samples, random_state=42)

    balanced_df.append(subset)

df = pd.concat(balanced_df)

print("\n✅ After balancing:\n")
print(df['attack_type'].value_counts())

# =========================
# 🔥 FEATURE ENGINEERING (UPGRADED)
# =========================
df['bytes_per_packet'] = df['total_bytes'] / (df['total_packets'] + 1e-6)
df['packets_per_second'] = df['total_packets'] / (df['flow_duration'] + 1e-6)

df['avg_packet_size'] = df['total_bytes'] / (df['total_packets'] + 1e-6)
df['byte_rate'] = df['total_bytes'] / (df['flow_duration'] + 1e-6)

df['burstiness'] = df['packets_per_second'] * df['avg_packet_size']

df['flag_sum'] = (
    df['syn_flag'] +
    df['ack_flag'] +
    df['rst_flag'] +
    df['psh_flag']
)

# =========================
# FEATURES + LABEL
# =========================
X = df.drop(['binary_label', 'attack_type'], axis=1)
y = df['attack_type']

X = X.astype(np.float32)

# =========================
# ENCODE LABELS
# =========================
le_attack = LabelEncoder()
y = le_attack.fit_transform(y)

print("\nNumber of classes:", len(le_attack.classes_))
print("Classes:", le_attack.classes_)

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 🔥 PIPELINE (UPGRADED)
# =========================
pipeline_attack = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(mutual_info_classif, k=15)),  # improved
    ('model', XGBClassifier(
        n_estimators=300,
        max_depth=9,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric='mlogloss',
        n_jobs=-1,
        tree_method='hist'
    ))
])

# =========================
# TRAIN
# =========================
print("\n🚀 Training Attack Classifier (Improved)...\n")

pipeline_attack.fit(X_train, y_train)

# =========================
# EVALUATE
# =========================
y_pred = pipeline_attack.predict(X_test)

print("\n=== ATTACK CLASSIFICATION REPORT ===\n")
print(classification_report(y_test, y_pred))

# =========================
# SAVE
# =========================
joblib.dump(pipeline_attack, "../models/attack_pipeline.pkl")
joblib.dump(le_attack, "../models/attack_label_encoder.pkl")

print("\n✅ Attack model saved successfully")

Attack dataset shape: (4294063, 18)

✅ Final attack classes:

attack_type
DDoS          2537036
DoS            961142
IoT            402282
Recon          197267
MITM            44658
Other           28535
BruteForce      10908
WebAttack        2948
Botnet           1931
Malware           487
Name: count, dtype: int64

✅ After balancing:

attack_type
DDoS          150000
Recon         150000
DoS           150000
IoT           150000
MITM           44658
Other          28535
BruteForce     10908
WebAttack       2948
Botnet          1931
Malware          487
Name: count, dtype: int64

Number of classes: 10
Classes: ['Botnet' 'BruteForce' 'DDoS' 'DoS' 'IoT' 'MITM' 'Malware' 'Other' 'Recon'
 'WebAttack']

🚀 Training Attack Classifier (Improved)...


=== ATTACK CLASSIFICATION REPORT ===

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       386
           1       0.93      0.81      0.87      2182
           2       0.80      0.77      0.79

In [16]:
print("Columns:", X.columns.tolist())
print("Total features:", len(X.columns))

Columns: ['dst_port', 'protocol', 'flow_duration', 'total_packets', 'total_bytes', 'min_pkt_len', 'max_pkt_len', 'avg_pkt_len', 'pkt_len_std', 'flow_rate', 'iat', 'syn_flag', 'ack_flag', 'rst_flag', 'psh_flag', 'ttl', 'bytes_per_packet', 'packets_per_second', 'avg_packet_size', 'byte_rate', 'burstiness', 'flag_sum']
Total features: 22
